In [35]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.metrics import f1_score, accuracy_score

In [36]:
df = pd.DataFrame([[0, 1],
                   [1, 1],
                   [2, 1],
                   [3, -1],
                   [4, -1],
                   [5, -1],
                   [6, 1],
                   [7, 1],
                   [8, 1],
                   [9, -1]])

X = df.iloc[:, :-1]
Y = df.iloc[:, -1]

### 第一个弱学习器

In [37]:
## 初始化样本权重
w1 = np.ones(df.shape[0]) / df.shape[0]
print(w1)
print(df.shape)
# sys.exit()

[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
(10, 2)


In [38]:
##构造弱分类器G1
model1 = DecisionTreeClassifier(max_depth=1)
model1.fit(X, Y, sample_weight=w1)

DecisionTreeClassifier(max_depth=1)

In [39]:
###误差率
print(model1.predict(X))
# print(model1.predict(X)!=Y)
# print(w1[model1.predict(X)!=Y])
e1 = sum(w1[model1.predict(X) != Y])
print(e1)

[ 1  1  1 -1 -1 -1 -1 -1 -1 -1]
0.30000000000000004


In [40]:
###弱学习器G1的权重α1
a1 = 0.5 * np.log((1 - e1) / e1)
print(a1)

0.4236489301936017


In [41]:
# f=a1*model1
y_hat = np.sign(a1 * model1.predict(X))
print(Y.tolist())
print(y_hat)

[1, 1, 1, -1, -1, -1, 1, 1, 1, -1]
[ 1.  1.  1. -1. -1. -1. -1. -1. -1. -1.]


### 第二个弱学习器G2

In [42]:
print(Y * model1.predict(X))
print(-a1 * Y * model1.predict(X))

0    1
1    1
2    1
3    1
4    1
5    1
6   -1
7   -1
8   -1
9    1
Name: 1, dtype: int64
0   -0.423649
1   -0.423649
2   -0.423649
3   -0.423649
4   -0.423649
5   -0.423649
6    0.423649
7    0.423649
8    0.423649
9   -0.423649
Name: 1, dtype: float64


In [43]:
###更新样本权重值
w2 = w1 * np.exp(-a1 * Y * model1.predict(X))
print(w2)
w2 = np.array(w2 / sum(w2))  ##归一化
print(w2)

0    0.065465
1    0.065465
2    0.065465
3    0.065465
4    0.065465
5    0.065465
6    0.152753
7    0.152753
8    0.152753
9    0.065465
Name: 1, dtype: float64
[0.07142857 0.07142857 0.07142857 0.07142857 0.07142857 0.07142857
 0.16666667 0.16666667 0.16666667 0.07142857]


In [44]:
##训练模型G2
model2 = DecisionTreeClassifier(max_depth=1)
model2.fit(X, Y, sample_weight=w2)
print(model2.predict(X))
# print(model2.predict(X)!=Y)
# print(w2[model2.predict(X)!=Y])

[ 1  1  1  1  1  1  1  1  1 -1]


In [45]:
###误差率e2
e2 = sum(w2[model2.predict(X) != Y])
print(e2)

0.21428571428571427


In [46]:
###求G2的权重α2
a2 = 0.5 * np.log((1 - e2) / e2)
print(a2)

0.6496414920651304


In [47]:
# f = a1*G1+a2*G2
y_hat = np.sign(a1 * model1.predict(X) + a2 * model2.predict(X))
print(Y.tolist())
print(y_hat)

[1, 1, 1, -1, -1, -1, 1, 1, 1, -1]
[ 1.  1.  1.  1.  1.  1.  1.  1.  1. -1.]


### 第三个弱学习器 G3

In [48]:
###更新样本权重值
# w3 = w1 * np.exp(-a1 * Y * model1.predict(X))* np.exp(-a2 * Y * model2.predict(X))
# print(w3)
w3 = w2 * np.exp(-a2 * Y * model2.predict(X))
print(w3)
w3 = np.array(w3 / sum(w3))  ##归一化
print(w3)

0    0.037302
1    0.037302
2    0.037302
3    0.136775
4    0.136775
5    0.136775
6    0.087039
7    0.087039
8    0.087039
9    0.037302
Name: 1, dtype: float64
[0.04545455 0.04545455 0.04545455 0.16666667 0.16666667 0.16666667
 0.10606061 0.10606061 0.10606061 0.04545455]


In [49]:
###训练模型G3
model3 = DecisionTreeClassifier(max_depth=1)
model3.fit(X, Y, sample_weight=w3)

DecisionTreeClassifier(max_depth=1)

In [50]:
###误差率e3
e3 = sum(w3[model3.predict(X) != Y])
print(e3)

0.18181818181818185


In [51]:
###求G3的权重α3
a3 = 0.5 * np.log((1 - e3) / e3)
print(a3)
# f = a1*G1+a2*G2+a3*G3

0.752038698388137


In [52]:
##最终分类器的线性组合f3
# f3 =a1*model1+a2*model2+a3*model3
## 最终的分类器G
# G = sign(f3)
##预测
print(model3.predict(X))

y_hat = np.sign(a1 * model1.predict(X) + a2 * model2.predict(X) + a3 * model3.predict(X))
print(Y.tolist())
print(y_hat)

[-1 -1 -1 -1 -1 -1  1  1  1  1]
[1, 1, 1, -1, -1, -1, 1, 1, 1, -1]
[ 1.  1.  1. -1. -1. -1.  1.  1.  1. -1.]
